# Multi-image in-context learning
Seed Isaac with cat vs. dog exemplars and detect the correct class in a new frame.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericpence/perceptron_repo/blob/main/cookbook/recipes/capabilities/multi-image-in-context-learning/multi-image-in-context-learning.ipynb)

In [ ]:
!uv pip install --upgrade perceptron pillow

## Configure the SDK and resolve assets

In [ ]:
import os
from pathlib import Path
from urllib.request import urlretrieve

from IPython.display import Image as IPyImage, display
from PIL import Image, ImageDraw, ImageFont

from perceptron import annotate_image, bbox, configure, detect
from perceptron.pointing.geometry import scale_box_to_pixels

PERCEPTRON_API_KEY = os.environ.get("PERCEPTRON_API_KEY", "<your Perceptron API key>")

configure(
    provider="perceptron",
    api_key=PERCEPTRON_API_KEY,
)

BASE_URL = "https://raw.githubusercontent.com/perceptron-ai-inc/perceptron/main/cookbook/_shared/assets/in-context-learning/multi/"
CAT_IMAGE = Path("classA.jpg")
DOG_IMAGE = Path("classB.webp")
TARGET_IMAGE = Path("cat_dog_input.png")
for filename, path_obj in (("classA.jpg", CAT_IMAGE), ("classB.webp", DOG_IMAGE), ("cat_dog_input.png", TARGET_IMAGE)):
    if not path_obj.exists():
        urlretrieve(BASE_URL + filename, path_obj)


> Exemplar shots live in `cookbook/_shared/assets/in-context-learning/multi/` so scripts and notebooks stay in sync. Perceptron returns all geometry on the normalized 1–1000 grid—convert the coordinates back to pixels (as in the rendering cell) before drawing overlays or computing metrics.


## Build the exemplar shots

In [ ]:
cat_example = annotate_image(
    str(CAT_IMAGE),
    {
        "classA": [
            bbox(316, 136, 703, 906, mention="classA"),
        ]
    },
)

dog_example = annotate_image(
    str(DOG_IMAGE),
    {
        "classB": [
            bbox(161, 48, 666, 980, mention="classB"),
        ]
    },
)

## Detect the target frame

In [ ]:
result = detect(
    str(TARGET_IMAGE),
    classes=["classA", "classB"],
    examples=[cat_example, dog_example],
)

print(result.text)
boxes = result.points or []
for box in boxes:
    print(box)

## Preview the annotated output

In [ ]:
img = Image.open(TARGET_IMAGE).convert("RGB")
draw = ImageDraw.Draw(img)
try:
    font = ImageFont.truetype("arial.ttf", size=20)
except OSError:
    font = ImageFont.load_default()

for box in boxes:
    scaled = scale_box_to_pixels(box, width=img.width, height=img.height)
    top_left = scaled.top_left
    bottom_right = scaled.bottom_right
    tlx, tly = int(round(top_left.x)), int(round(top_left.y))
    brx, bry = int(round(bottom_right.x)), int(round(bottom_right.y))
    draw.rectangle([tlx, tly, brx, bry], outline="lime", width=3)
    label = box.mention or getattr(box, "label", None) or box.mention
    text_position = (tlx, max(tly - 20, 0))
    draw.text(text_position, label, fill="lime", font=font)

annotated_path = TARGET_IMAGE.with_name(f"{TARGET_IMAGE.stem}_annotated.png")
img.save(annotated_path)
print(f"Saved annotated target to {annotated_path}")
display(IPyImage(filename=str(annotated_path)))


## Conclusion & next steps
- Add more exemplar shots (or additional classes) to make tougher distinctions.
- Vary the prompt or classes to localize other objects across multiple example images.
- Pair this workflow with the single-image ICL or detection notebooks to compare approaches.